# Physiology–MWL correlations

Exploratory Spearman associations between the final physiological features and perceived Mental Workload (MWL), overall and separately for Haptic and NoHA participants.

This analysis uses pairwise-complete observations and does **not** account for repeated observations within participants. It is descriptive and complementary; it is not a replacement for a later repeated-measures model. Separate group correlations and their difference are not a formal test of group interaction.

## 1. Imports

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

## 2. Configuration

In [ ]:
CORRELATION_METHOD = "spearman"
INCLUDE_IMPUTED_MWL = True
FDR_ALPHA = 0.05
SAVE_FIGURES = True
SAVE_TABLES = True
TOP_N = 10
N_EXTREME_LABELS = 6

assert CORRELATION_METHOD == "spearman", "This notebook intentionally implements Spearman correlation only."

## 3. Repository and analysis paths

In [ ]:
def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        authoritative = candidate / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
        if authoritative.exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current directory")

REPO_ROOT = find_repository_root()
INPUT_PATH = REPO_ROOT / "outputs/final_features/physiology_mwl_analysis_dataset.csv"
FIGURE_DIR = REPO_ROOT / "postprocessing/outputs/figures"
TABLE_DIR = REPO_ROOT / "postprocessing/outputs/tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CORRELATION_TABLE_PATH = TABLE_DIR / "physiology_mwl_spearman_correlations.csv"
HEATMAP_PATH = FIGURE_DIR / "physiology_mwl_correlation_heatmap.png"
GROUP_SCATTER_PATH = FIGURE_DIR / "haptic_vs_noha_mwl_correlations.png"
print("Input:", INPUT_PATH)

## 4. Load and validate

Physiological columns are selected only through the finalized analysis prefixes. Metadata and other numeric columns are never treated as physiological features.

In [ ]:
all_data = pd.read_csv(INPUT_PATH)
MODALITY_PREFIXES = {
    "ECG": "delta_ecg_",
    "EDA": "delta_eda_",
    "RESP": "delta_resp_",
    "TEMP": "delta_temp_",
    "fNIRS": "fnirs_",
}
modality_features = {
    modality: [column for column in all_data.columns if column.startswith(prefix)]
    for modality, prefix in MODALITY_PREFIXES.items()
}
feature_columns = [feature for modality in MODALITY_PREFIXES for feature in modality_features[modality]]
feature_to_modality = {feature: modality for modality, features in modality_features.items() for feature in features}

assert len(feature_columns) == 79, f"Expected exactly 79 physiological features, found {len(feature_columns)}"
assert len(set(feature_columns)) == 79
assert not all_data.duplicated(["participant_id", "phase", "block_index"]).any()
assert set(all_data.group) == {"Haptic", "NoHA"}

data = all_data.copy() if INCLUDE_IMPUTED_MWL else all_data.loc[all_data.mwl_source.ne("imputed_previous")].copy()
validation_summary = pd.DataFrame({
    "metric": ["observations", "participants", "Haptic participants", "NoHA participants", "physiological features", "missing MWL"],
    "value": [len(data), data.participant_id.nunique(), data.loc[data.group.eq("Haptic"), "participant_id"].nunique(), data.loc[data.group.eq("NoHA"), "participant_id"].nunique(), len(feature_columns), data.mwl_value.isna().sum()],
})
observations_by_group = data.groupby("group", observed=True).size().rename("observations")
observations_by_phase = data.groupby("phase", observed=True).size().rename("observations")
modality_missingness = pd.DataFrame([
    {
        "modality": modality,
        "n_features": len(features),
        "missing_cells": int(data[features].isna().sum().sum()),
        "missing_cell_fraction": float(data[features].isna().mean().mean()),
        "rows_with_modality_entirely_missing": int(data[features].isna().all(axis=1).sum()),
    }
    for modality, features in modality_features.items()
])
print(validation_summary.to_string(index=False))
print("\nObservations by group:\n", observations_by_group.to_string())
print("\nObservations by phase:\n", observations_by_phase.to_string())
print("\nModality missingness:\n", modality_missingness.to_string(index=False))

## 5. Correlation and FDR helpers

Constant and insufficient features are retained with an explicit status and undefined correlation statistics. Benjamini–Hochberg correction is applied only to finite p-values.

In [ ]:
def benjamini_hochberg(p_values):
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return adjusted
    finite_values = values[finite]
    order = np.argsort(finite_values)
    ranked = finite_values[order]
    m = len(ranked)
    ranked_adjusted = ranked * m / np.arange(1, m + 1)
    ranked_adjusted = np.minimum.accumulate(ranked_adjusted[::-1])[::-1]
    ranked_adjusted = np.clip(ranked_adjusted, 0, 1)
    restored = np.empty(m, dtype=float)
    restored[order] = ranked_adjusted
    adjusted[np.flatnonzero(finite)] = restored
    return adjusted


def correlate_feature(frame, feature, family):
    paired = frame[["participant_id", feature, "mwl_value"]].dropna()
    n_observations = len(paired)
    n_participants = paired.participant_id.nunique()
    if n_observations < 3 or n_participants < 2:
        return {"family": family, "feature": feature, "modality": feature_to_modality[feature], "n_observations": n_observations, "n_participants": n_participants, "rho": np.nan, "p_value": np.nan, "status": "insufficient_data"}
    if paired[feature].nunique() < 2:
        return {"family": family, "feature": feature, "modality": feature_to_modality[feature], "n_observations": n_observations, "n_participants": n_participants, "rho": np.nan, "p_value": np.nan, "status": "constant_feature"}
    if paired.mwl_value.nunique() < 2:
        return {"family": family, "feature": feature, "modality": feature_to_modality[feature], "n_observations": n_observations, "n_participants": n_participants, "rho": np.nan, "p_value": np.nan, "status": "constant_mwl"}
    result = spearmanr(paired[feature], paired.mwl_value, nan_policy="omit")
    return {"family": family, "feature": feature, "modality": feature_to_modality[feature], "n_observations": n_observations, "n_participants": n_participants, "rho": float(result.statistic), "p_value": float(result.pvalue), "status": "ok"}


def correlation_family(frame, family):
    result = pd.DataFrame([correlate_feature(frame, feature, family) for feature in feature_columns])
    result["p_fdr"] = benjamini_hochberg(result.p_value)
    result["significant_nominal"] = result.p_value.lt(FDR_ALPHA)
    result["significant_fdr"] = result.p_fdr.lt(FDR_ALPHA)
    return result

## 6. Spearman correlations: ALL, Haptic, and NoHA

In [ ]:
families = {
    "all": data,
    "haptic": data.loc[data.group.eq("Haptic")],
    "noha": data.loc[data.group.eq("NoHA")],
}
long_results = pd.concat([correlation_family(frame, family) for family, frame in families.items()], ignore_index=True)
family_diagnostics = long_results.groupby(["family", "status"], observed=True).size().rename("n_features").reset_index()
print(family_diagnostics.to_string(index=False))

## 7. Master correlation table

In [ ]:
master = pd.DataFrame({"feature": feature_columns, "modality": [feature_to_modality[f] for f in feature_columns]})
for family in families:
    subset = long_results.loc[long_results.family.eq(family)].set_index("feature")
    for source, target in [
        ("rho", f"rho_{family}"), ("p_value", f"p_{family}"), ("p_fdr", f"p_fdr_{family}"),
        ("n_observations", f"n_{family}"), ("n_participants", f"n_participants_{family}"),
        ("significant_nominal", f"significant_nominal_{family}"), ("significant_fdr", f"significant_fdr_{family}"),
        ("status", f"status_{family}"),
    ]:
        master[target] = master.feature.map(subset[source])
master["delta_rho"] = master.rho_haptic - master.rho_noha
master["abs_delta_rho"] = master.delta_rho.abs()
if SAVE_TABLES:
    master.to_csv(CORRELATION_TABLE_PATH, index=False)
print("Saved:", CORRELATION_TABLE_PATH)
print("Master shape:", master.shape)

## 8. Ranked exploratory summaries

Rankings use absolute correlation magnitude. Nominal and FDR p-values are shown but are not used to discard features.

In [ ]:
def ranked_table(family, n=TOP_N):
    columns = ["feature", "modality", f"rho_{family}", f"p_{family}", f"p_fdr_{family}", f"n_{family}"]
    return master.loc[master[f"rho_{family}"].notna(), columns].assign(abs_rho=lambda x: x[f"rho_{family}"].abs()).sort_values("abs_rho", ascending=False).drop(columns="abs_rho").head(n)

top_all = ranked_table("all")
top_haptic = ranked_table("haptic")
top_noha = ranked_table("noha")
largest_group_differences = master.loc[master.abs_delta_rho.notna(), ["feature", "modality", "rho_haptic", "p_haptic", "p_fdr_haptic", "n_haptic", "rho_noha", "p_noha", "p_fdr_noha", "n_noha", "delta_rho", "abs_delta_rho"]].sort_values("abs_delta_rho", ascending=False).head(TOP_N)
print("TOP ALL:\n", top_all.to_string(index=False))
print("\nTOP HAPTIC:\n", top_haptic.to_string(index=False))
print("\nTOP NOHA:\n", top_noha.to_string(index=False))
print("\nLARGEST DESCRIPTIVE HAPTIC–NOHA DIFFERENCES (not interaction tests):\n", largest_group_differences.to_string(index=False))

## 9. Modality-level summary

In [ ]:
modality_rows = []
for modality, subset in master.groupby("modality", sort=False):
    row = {"modality": modality, "n_features": len(subset)}
    for family in families:
        row[f"n_nominal_{family}"] = int(subset[f"significant_nominal_{family}"].sum())
        row[f"n_fdr_{family}"] = int(subset[f"significant_fdr_{family}"].sum())
        valid = subset.dropna(subset=[f"rho_{family}"])
        if len(valid):
            strongest = valid.loc[valid[f"rho_{family}"].abs().idxmax()]
            row[f"strongest_feature_{family}"] = strongest.feature
            row[f"strongest_rho_{family}"] = strongest[f"rho_{family}"]
        else:
            row[f"strongest_feature_{family}"] = ""
            row[f"strongest_rho_{family}"] = np.nan
    modality_rows.append(row)
modality_summary = pd.DataFrame(modality_rows)
print(modality_summary.to_string(index=False))
if SAVE_TABLES:
    modality_summary.to_csv(TABLE_DIR / "physiology_mwl_modality_summary.csv", index=False)

## 10. Correlation heatmap

In [ ]:
rho_columns = ["rho_all", "rho_haptic", "rho_noha"]
heatmap_values = master[rho_columns].to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(7.5, 22))
image = ax.imshow(heatmap_values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(3), ["ALL", "Haptic", "NoHA"])
ax.set_yticks(np.arange(len(master)), master.feature, fontsize=5)
ax.set_title("Spearman associations between physiological features and MWL")

boundaries = np.cumsum([len(modality_features[m]) for m in MODALITY_PREFIXES])[:-1]
for boundary in boundaries:
    ax.axhline(boundary - 0.5, color="black", linewidth=1.2)
starts = np.r_[0, boundaries]
ends = np.r_[boundaries, len(master)]
for modality, start, end in zip(MODALITY_PREFIXES, starts, ends):
    ax.text(-0.62, (start + end - 1) / 2, modality, transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=8, fontweight="bold")
colorbar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.03)
colorbar.set_label("Spearman rho")
fig.subplots_adjust(left=0.46, right=0.92, top=0.98, bottom=0.03)
if SAVE_FIGURES:
    fig.savefig(HEATMAP_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 11. Descriptive Haptic versus NoHA comparison

Distance from the identity line is descriptive only; no formal comparison of correlations is performed.

In [ ]:
modality_colors = {"ECG": "tab:red", "EDA": "tab:green", "RESP": "tab:blue", "TEMP": "tab:orange", "fNIRS": "tab:purple"}
fig, ax = plt.subplots(figsize=(7, 7))
for modality in MODALITY_PREFIXES:
    subset = master.loc[master.modality.eq(modality)]
    ax.scatter(subset.rho_noha, subset.rho_haptic, s=38, alpha=0.75, label=modality, color=modality_colors[modality])
ax.axhline(0, color="grey", linewidth=1)
ax.axvline(0, color="grey", linewidth=1)
ax.plot([-1, 1], [-1, 1], linestyle="--", color="black", linewidth=1, label="y = x")
for _, row in master.nlargest(N_EXTREME_LABELS, "abs_delta_rho").iterrows():
    ax.annotate(row.feature, (row.rho_noha, row.rho_haptic), xytext=(4, 4), textcoords="offset points", fontsize=7)
ax.set(xlim=(-1, 1), ylim=(-1, 1), xlabel="Spearman rho: NoHA", ylabel="Spearman rho: Haptic", title="Haptic versus NoHA physiological–MWL correlations")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2); ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(GROUP_SCATTER_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 12. Sensitivity to the approved imputed MWL rating

This comparison changes only inclusion of the single `imputed_previous` MWL observation. Primary results are not overwritten.

In [ ]:
all_with_imputation = correlation_family(all_data, "with_imputation").set_index("feature")
all_observed_only = correlation_family(all_data.loc[all_data.mwl_source.ne("imputed_previous")], "observed_only").set_index("feature")
sensitivity = pd.DataFrame({
    "feature": feature_columns,
    "modality": [feature_to_modality[f] for f in feature_columns],
    "rho_with_imputation": [all_with_imputation.loc[f, "rho"] for f in feature_columns],
    "rho_observed_only": [all_observed_only.loc[f, "rho"] for f in feature_columns],
})
sensitivity["absolute_change_rho"] = (sensitivity.rho_with_imputation - sensitivity.rho_observed_only).abs()
finite_changes = sensitivity.absolute_change_rho.dropna()
max_change = finite_changes.max()
median_change = finite_changes.median()
largest_change_row = sensitivity.loc[sensitivity.absolute_change_rho.idxmax()]
print(f"Maximum absolute change in rho: {max_change:.6g}")
print(f"Median absolute change in rho: {median_change:.6g}")
print("Largest-change feature:\n", largest_change_row.to_string())
if SAVE_TABLES:
    sensitivity.to_csv(TABLE_DIR / "physiology_mwl_imputation_sensitivity.csv", index=False)

## 13. Automated exploratory summary

In [ ]:
print("PHYSIOLOGY–MWL EXPLORATORY CORRELATION SUMMARY")
print(f"- features analyzed: {len(master)}")
for family, label in [("all", "ALL"), ("haptic", "Haptic"), ("noha", "NoHA")]:
    print(f"- {label}: nominal p < {FDR_ALPHA}: {int(master[f'significant_nominal_{family}'].sum())}; FDR < {FDR_ALPHA}: {int(master[f'significant_fdr_{family}'].sum())}")
print("- strongest correlations by modality:")
for _, row in modality_summary.iterrows():
    print(f"  {row.modality}: ALL {row.strongest_feature_all} (rho={row.strongest_rho_all:.3f}); Haptic {row.strongest_feature_haptic} (rho={row.strongest_rho_haptic:.3f}); NoHA {row.strongest_feature_noha} (rho={row.strongest_rho_noha:.3f})")
print("- largest descriptive Haptic–NoHA rho differences (not statistically tested):")
for _, row in largest_group_differences.head(5).iterrows():
    print(f"  {row.feature}: delta_rho={row.delta_rho:.3f}")
print(f"- excluding the imputed MWL: maximum |change in rho|={max_change:.6g}; median={median_change:.6g}")
print("- 'associated with MWL' here means exploratory correlation; whether associations differ between groups has NOT been statistically tested.")